In [1]:
# ============================================================================
# notebook: 02_indices.ipynb
# Project: "Incidental vs. Engineered Approval" — cross-group audit of approval quality
# Stage 2: full implementation of the three reliability indices + axis re-verdict.
#   Density   : group-relative neighbor density in the audit space (Problem 3 fix).
#   Stability : REAL TreeExplainer SHAP — neighbor-variance above a bootstrap
#               noise floor (Problem 6 fix; replaces Stage-0 proxy).
#   Fragility : perturbation sensitivity as Δp, a common output unit (Problem 6 fix).
#   Then: recompute the 3-index correlation on the borderline cohort and re-issue
#         the axis-independence verdict (R-2 decision point).
# Depends on artifacts written by 01_cohort_and_model.ipynb.
# Requires: pip install shap
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports, config, load Stage-1 artifacts
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import joblib
import time

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

AXIS_CORR_THRESHOLD = 0.70   # pre-registered: max |r| above this => drop to 2 axes
K_NEIGHBORS = 20             # neighborhood size for density & stability
SIGMA_FRAC = 0.10            # perturbation sd (standardized space => 1 sd = 1.0)
FRAG_REPS = 30               # perturbation repetitions
NOISE_FLOOR_SEEDS = 10       # bootstrap seeds for SHAP noise floor (kept at 10)

df = pd.read_parquet("stage1_cohort.parquet")
X_scaled = np.load("stage1_X_audit_scaled.npy")
approved_idx = np.load("stage1_approved_idx.npy")
borderline_idx = np.load("stage1_borderline_idx.npy")
scaler = joblib.load("stage1_scaler.joblib")
rf = joblib.load("stage1_rf_final.joblib")

AUDIT_AXIS = (["LIMIT_BAL"]
              + [f"BILL_AMT{i}" for i in range(1, 7)]
              + [f"PAY_AMT{i}" for i in range(1, 7)])

print(f"Loaded. Approved={len(approved_idx):,}  Borderline={len(borderline_idx):,}")
print(f"Audit axis dims: {X_scaled.shape[1]}")

# Row-index -> positional-index map (X_scaled rows align with df row order)
pos_of = {ridx: i for i, ridx in enumerate(df.index.to_numpy())}
appr_pos = np.array([pos_of[r] for r in approved_idx])
bord_pos = np.array([pos_of[r] for r in borderline_idx])


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Import SHAP
# ─────────────────────────────────────────────────────────────────────────
try:
    import shap
    print("shap", shap.__version__)
except ImportError:
    raise ImportError("Run `pip install shap` in this environment, then re-run.")

def shap_class1(sv):
    """Return the class-1 SHAP matrix (n, features) across shap version formats."""
    if isinstance(sv, list):              # older: [class0, class1]
        return np.asarray(sv[1])
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv   # shap>=0.4x binary: (n, feat, class)


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — DENSITY (group-relative). Problem 3 fix.
# Raw density = inverse mean distance to k nearest neighbors within the AUDIT
# space, among APPROVED cases; then normalized WITHIN each group to a percentile
# so a small group is not automatically "low density" for sample-size reasons.
# Computed over ALL approved cases so the within-group percentile stays defined.
# ─────────────────────────────────────────────────────────────────────────
X_appr = X_scaled[appr_pos]
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1).fit(X_appr)
dist, nbr_idx_full = knn.kneighbors(X_appr)          # keep neighbor idx for reuse
density_raw = 1.0 / (dist[:, 1:].mean(axis=1) + 1e-9)  # drop self (col 0)

dens = pd.Series(density_raw, index=approved_idx, name="density_raw")
grp = df.loc[approved_idx, "GROUP"]
density_grouprel = dens.groupby(grp).rank(pct=True)   # within-group percentile [0,1]
density_grouprel.name = "density_grouprel"
print("Density computed (raw + group-relative), over all approved cases.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — STABILITY via REAL SHAP, noise-floor corrected. Problem 6 fix.
# (a) SHAP once over all approved (neighbors may lie outside the borderline set).
# (b) instability = mean L2 distance of a point's SHAP vector to its k neighbors'
#     SHAP vectors — VECTORIZED (no per-point python loop).
# (c) noise floor = SHAP movement across RF refits (10 seeds, unchanged); signal
#     = instability ABOVE the per-point floor.
# check_additivity=False skips SHAP's internal post-hoc verification only; it does
# not change the SHAP values.
# ─────────────────────────────────────────────────────────────────────────
# (a)
expl = shap.TreeExplainer(rf)
shap_appr = shap_class1(expl.shap_values(X_appr, check_additivity=False))
print("SHAP matrix (approved):", shap_appr.shape)

# (b) vectorized neighbor SHAP distance
nbr = nbr_idx_full[:, 1:]                              # (n, k), drop self
diff = shap_appr[nbr] - shap_appr[:, None, :]         # (n, k, features)
instability = np.linalg.norm(diff, axis=2).mean(axis=1)   # (n,)

# (c) noise floor across seeds
y_all = df["VIP_CLEAR"].values
floor_shaps = []
t0 = time.time()
for s in range(NOISE_FLOOR_SEEDS):
    rf_s = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                  random_state=1000 + s, n_jobs=-1).fit(X_scaled, y_all)
    sv_s = shap_class1(shap.TreeExplainer(rf_s).shap_values(X_appr, check_additivity=False))
    floor_shaps.append(sv_s)
    print(f"  noise-floor seed {s+1}/{NOISE_FLOOR_SEEDS} done ({time.time()-t0:.0f}s elapsed)")

floor_stack = np.stack(floor_shaps, axis=0)           # (seeds, n, features)
noise_floor = np.linalg.norm(floor_stack.std(axis=0), axis=1)   # per-point seed sd magnitude

instab_signal = np.clip(instability - noise_floor, 0, None)
stability = 1.0 / (instab_signal + 1e-9)              # higher = more stable

floor_ratio = (noise_floor / (instability + 1e-9)).mean()
print(f"\nMean raw instability   = {instability.mean():.4f}")
print(f"Mean SHAP noise floor  = {noise_floor.mean():.4f}")
print(f"Floor / instability    = {floor_ratio:.2%}  (share of raw instability that is model noise)")
print("NOTE: if the floor eats most of the signal, Stability is fragile -> weigh in verdict.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — FRAGILITY as Δp under perturbation. Problem 6 fix.
# All audit vars are continuous/quasi-continuous (PAY_* excluded as label axis).
# Batched: all reps in ONE predict_proba call. Same sigma/reps/seed => same result.
# ─────────────────────────────────────────────────────────────────────────
def fragility_dp_batched(model, X, sigma, reps, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    base = model.predict_proba(X)[:, 1]
    n, d = X.shape
    noise = rng.normal(0, sigma, size=(reps, n, d))
    Xp = (X[None, :, :] + noise).reshape(reps * n, d)
    p = model.predict_proba(Xp)[:, 1].reshape(reps, n)
    return np.abs(p - base[None, :]).mean(axis=0)

frag_appr = fragility_dp_batched(rf, X_appr, sigma=SIGMA_FRAC, reps=FRAG_REPS)
nonfragility = 1.0 - (frag_appr / (frag_appr.max() + 1e-9))
print(f"Fragility (Δp) computed. mean Δp = {frag_appr.mean():.4f}, max = {frag_appr.max():.4f}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Assemble the index table for APPROVED cases
# ─────────────────────────────────────────────────────────────────────────
idx_df = pd.DataFrame({
    "GROUP": df.loc[approved_idx, "GROUP"].values,
    "CELL": df.loc[approved_idx, "CELL"].values,
    "is_borderline": df.loc[approved_idx, "VIP_BORDERLINE_s1"].values,
    "Density": density_grouprel.values,
    "Stability": stability,
    "NonFragility": nonfragility,
}, index=approved_idx)

for c in ["Density", "Stability", "NonFragility"]:
    idx_df[c + "_pct"] = idx_df[c].rank(pct=True)

idx_df.to_parquet("stage2_indices_approved.parquet")
print("Saved stage2_indices_approved.parquet")
print(idx_df[["Density", "Stability", "NonFragility"]].describe().round(3).to_string())


# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — AXIS-INDEPENDENCE RE-VERDICT (R-2). The key decision of Stage 2.
# Recompute the 3-index correlation, now with REAL SHAP stability, on the
# BORDERLINE cohort (the RQ2/RQ3 analysis set).
# ─────────────────────────────────────────────────────────────────────────
bmask = idx_df["is_borderline"] == 1
B = idx_df.loc[bmask, ["Density", "Stability", "NonFragility"]]

corr = B.corr().abs()
max_off = corr.where(~np.eye(3, dtype=bool)).max().max()

print(f"Borderline cohort size for correlation: {bmask.sum()}")
print("\nAbsolute correlation of the three indices (REAL SHAP, borderline cohort):")
print(corr.round(3).to_string())
print(f"\nMax off-diagonal |r| = {max_off:.3f}  (cutoff = {AXIS_CORR_THRESHOLD}; Stage-0 proxy was 0.394)")

if max_off <= AXIS_CORR_THRESHOLD:
    axis_decision = "KEEP_3"
    print("\n>>> R-2 VERDICT: PASS — real-SHAP stability stays independent. Keep all 3 axes.")
else:
    keep = corr.sum().nsmallest(2).index.tolist()
    axis_decision = f"DROP_TO_2:{keep}"
    print(f"\n>>> R-2 VERDICT: BRANCH — axes too collinear under real SHAP. "
          f"Drop to 2 least-correlated: {keep}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Stage 2 summary
# ─────────────────────────────────────────────────────────────────────────
print("=" * 66)
print("STAGE 2 — INDICES IMPLEMENTED + AXIS RE-VERDICT")
print("=" * 66)
print(f"Density   : group-relative percentile (small-group artifact removed)")
print(f"Stability : real TreeExplainer SHAP, noise-floor corrected")
print(f"            (floor ate {floor_ratio:.1%} of raw instability)")
print(f"Fragility : Δp under sigma={SIGMA_FRAC} perturbation, {FRAG_REPS} reps")
print(f"Axis corr : max |r| = {max_off:.3f} vs {AXIS_CORR_THRESHOLD}  => {axis_decision}")
print("-" * 66)
if axis_decision == "KEEP_3":
    print("NEXT: Stage 3 builds the ensemble EngineeredScore and runs the")
    print("      predicted-prob-controlled partial correlation (confidence defense).")
else:
    print("NEXT: Stage 3 builds the ensemble on the 2 surviving axes.")
print("=" * 66)

with open("stage2_axis_decision.txt", "w") as f:
    f.write(axis_decision + f"\nmax_off_r={max_off:.4f}\nfloor_ratio={floor_ratio:.4f}\n")
print("Saved stage2_axis_decision.txt")

Loaded. Approved=11,089  Borderline=1,141
Audit axis dims: 13
shap 0.49.1
Density computed (raw + group-relative), over all approved cases.
SHAP matrix (approved): (11089, 13)
  noise-floor seed 1/10 done (1904s elapsed)
  noise-floor seed 2/10 done (3801s elapsed)
  noise-floor seed 3/10 done (5703s elapsed)
  noise-floor seed 4/10 done (7606s elapsed)
  noise-floor seed 5/10 done (9511s elapsed)
  noise-floor seed 6/10 done (11422s elapsed)
  noise-floor seed 7/10 done (13319s elapsed)
  noise-floor seed 8/10 done (15213s elapsed)
  noise-floor seed 9/10 done (17115s elapsed)
  noise-floor seed 10/10 done (19009s elapsed)

Mean raw instability   = 0.1491
Mean SHAP noise floor  = 0.0110
Floor / instability    = 8.37%  (share of raw instability that is model noise)
NOTE: if the floor eats most of the signal, Stability is fragile -> weigh in verdict.
Fragility (Δp) computed. mean Δp = 0.3428, max = 0.7163
Saved stage2_indices_approved.parquet
         Density  Stability  NonFragility
co